# Structured streaming Demo

In [49]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime
from pyspark.sql.functions import avg

import pandas as pd


In [50]:
# initialize the spark session
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0") \
    .getOrCreate()


In [51]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


In [52]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

In [100]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = parsed_df \
    .withColumn(
        "event_time",
        from_unixtime(col("timestamp"))
    ) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot"),
         #adding wiki in the group by
        col("wiki")
    ) \
    .agg(
        count("*").alias("edit_count")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("wiki"),
        col("edit_count")
    )

In [101]:
tumbling_results = []

def save_tumbling_batch(batch_df, batch_id):
    # First 5 lines of each batch for notebook traceability.
    pdf = batch_df.limit(5).toPandas()
    pdf["batch_id"] = batch_id
    tumbling_results.append(pdf)

In [92]:
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

In [102]:
tumbling_query = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .foreachBatch(save_overlapping_batch) \
    .start()

In [103]:
tumbling_query.stop()

In [97]:
# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [98]:
streaming_query_df.stop()

In [99]:
streaming_query_df.isActive

False

In [77]:
    #adding text legth variation
over_counts_df = parsed_df \
    .withColumn(
        "event_time",
        from_unixtime(col("timestamp"))
    ) \
    .withColumn(
        "length_variation",
        col("length.new") - col("length.old")
    ) \
    .groupBy(
        window(
            col("event_time"),
            "1 hour",
            "30 minutes"
        ),
        col("wiki")
    ) \
    .agg(
        count("*").alias("edit_count"),
        avg("length_variation").alias("avg_length_variation")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("wiki"),
        col("edit_count"),
        col("avg_length_variation")
    )

In [78]:
overlapping_results = []

def save_overlapping_batch(batch_df, batch_id):
    pdf = batch_df.limit(5).toPandas()
    pdf["batch_id"] = batch_id
    overlapping_results.append(pdf)

In [79]:
overlapping_query = over_counts_df \
    .writeStream \
    .outputMode("complete") \
    .foreachBatch(save_overlapping_batch) \
    .start()

In [80]:
overlapping_query.stop()

In [81]:
overlapping_trace = pd.concat(overlapping_results, ignore_index=True)

print("===== OVERLAPPING WINDOWS =====")
display(overlapping_trace)

===== OVERLAPPING WINDOWS =====


,window_start,window_end,wiki,edit_count,avg_length_variation,batch_id
0,2026-09-23 11:00:00,2026-09-23 12:00:00,frwiki,2332,301.025729,0
1,2026-09-23 12:00:00,2026-09-23 13:00:00,frwiki,1257,272.371519,0
2,2026-09-23 14:00:00,2026-09-23 15:00:00,frwiki,1103,133.061650,0
3,2026-09-23 14:30:00,2026-09-23 15:30:00,frwiki,463,89.833693,0
4,2026-09-24 16:30:00,2026-09-24 17:30:00,frwiki,115,115.530435,0


In [10]:
sliding_query_df = over_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

In [ ]:
sliding_query_df.stop()

The Wikimedia stream is filtered before being sent to Kafka. In this experiment, only edit events from the French Wikipedia server (fr.wikipedia.org) are produced.